In [12]:
# !pip install seleniumbase beautifulsoup4 pandas nest-asyncio --upgrade

### Imports

In [13]:
import asyncio
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from pathlib import Path
import re
from seleniumbase import cdp_driver
from urllib.parse import urlencode

### Configuration

In [14]:
STATE_FILE = Path("indeed_storage_state.json")
BASE_URL = "https://ch-fr.indeed.com"

locations = ["Genève, GE", "Lausanne, VD"]

params = {
    "q": "",
    "l": locations[0],
    "radius": 25,
    "sort": "date",
    "limit": 240,
    "fromage": 1,
    "start": 0,
}

full_url = f"{BASE_URL}/jobs?{urlencode(params)}"
print(f"🔗 URL cible: {full_url}")

🔗 URL cible: https://ch-fr.indeed.com/jobs?q=&l=Gen%C3%A8ve%2C+GE&radius=25&sort=date&limit=240&fromage=1&start=0


In [15]:
JOB_SELECTORS = [
    '[data-testid="jobCard"]',
    '.job_seen_beacon',
    '.jobsearch-ResultsList > div',
    '.result',
]

CAPTCHA_MARKERS = [
    "captcha",
    "cf-turnstile",
    "just a moment",
    "verify you are human",
]


def find_job_cards(html):
    soup = BeautifulSoup(html, "html.parser")

    # Utiliser le premier sélecteur qui retourne des résultats
    for selector in JOB_SELECTORS:
        cards = soup.select(selector)
        if cards:
            return cards

    return []


async def scrape_indeed_with_scroll():
    driver = await cdp_driver.start_async(headless=False, xvfb=True)

    try:
        print(f"🌐 Navigation vers : {full_url}")
        page = await driver.get(full_url)
        await asyncio.sleep(3)

        # CAPTCHA + attente du rendu des offres
        for attempt in range(4):
            html = await page.get_content()
            job_cards = find_job_cards(html)

            if job_cards:
                print(f"✅ Page chargée : {len(job_cards)} offres détectées")
                break

            page_lower = html.lower()

            if any(marker in page_lower for marker in CAPTCHA_MARKERS):
                print(
                    f"🧩 CAPTCHA détecté "
                    f"(tentative {attempt + 1}/4)"
                )
                await page.solve_captcha()
                await asyncio.sleep(4)
            else:
                print("⏳ Page chargée, attente des offres...")
                await asyncio.sleep(2)

        # Scroll dans le DOM via CDP
        print("📜 Scroll...")

        for i in range(3):
            await page.evaluate(
                "window.scrollTo(0, document.body.scrollHeight)"
            )
            await asyncio.sleep(2)
            print(f"  Scroll {i + 1}/3 effectué")

        await page.evaluate("window.scrollTo(0, 0)")
        await asyncio.sleep(1)

        # Capture du DOM rendu, et non de l'ancienne variable html
        html = await page.get_content()
        job_cards = find_job_cards(html)

        print(f"📊 {len(job_cards)} offres chargées")

        if not job_cards:
            current_url = await page.get_current_url()
            print(f"⚠️ URL réellement capturée : {current_url}")

            await page.save_screenshot(
                filename="indeed_debug.png",
                full_page=True,
            )

        return html

    finally:
        driver.stop()


# Jupyter accepte directement await au niveau de la cellule
html = await scrape_indeed_with_scroll()

🌐 Navigation vers : https://ch-fr.indeed.com/jobs?q=&l=Gen%C3%A8ve%2C+GE&radius=25&sort=date&limit=240&fromage=1&start=0
🧩 CAPTCHA détecté (tentative 1/4)
✅ Page chargée : 51 offres détectées
📜 Scroll...
  Scroll 1/3 effectué
  Scroll 2/3 effectué
  Scroll 3/3 effectué
📊 51 offres chargées


### Parsing des offres

In [16]:
def parse_indeed_jobs_advanced(html):
    soup = BeautifulSoup(html, 'html.parser')
    jobs = []
    
    # Méthode 1: Via data-testid (le plus fiable)
    job_cards = soup.find_all('div', {'data-testid': 'jobCard'})
    
    if not job_cards:
        # Méthode 2: Fallback avec les classes CSS
        job_cards = soup.select('.job_seen_beacon, .jobsearch-ResultsList > div, .result')
    
    for card in job_cards:
        try:
            job = {}
            
            # Titre - plusieurs selecteurs possibles
            title_elem = (
                card.find('h2', {'data-testid': 'jobTitle'}) or 
                card.find('a', {'data-testid': 'jobTitle'}) or
                card.find('h3', {'class': 'jobTitle'}) or
                card.find('a', {'class': 'jcs-JobTitle'}) or
                card.find('span', {'id': re.compile(r'jobTitle-')})
            )
            job['titre'] = title_elem.text.strip() if title_elem else 'N/A'

            # Entreprise
            company_elem = (
                card.find('span', {'data-testid': 'companyName'}) or
                card.find('div', {'data-testid': 'companyName'}) or
                card.find('span', {'class': 'companyName'}) or
                card.find('span', {'data-testid': 'company-name'})
            )
            job['entreprise'] = company_elem.text.strip() if company_elem else 'N/A'
            
            # Lieu
            location_elem = (
                card.find('div', {'data-testid': 'location'}) or
                card.find('span', {'data-testid': 'location'}) or
                card.find('div', {'class': 'location'}) or
                card.find('div', {'data-testid': 'text-location'})
            )
            job['lieu'] = location_elem.text.strip() if location_elem else 'N/A'
            
            # Salaire
            salary_elem = (
                card.find('div', {'data-testid': 'salary'}) or
                card.find('span', {'data-testid': 'salary'}) or
                card.find('div', {'class': 'salary-snippet'}) or
                card.find('li', {'class': 'salary-snippet-container'})
            )
            job['salaire'] = salary_elem.text.strip() if salary_elem else 'N/A'
            
            # Date de publication
            date_elem = (
                card.find('span', {'data-testid': 'postDate'}) or
                card.find('div', {'data-testid': 'postDate'}) or
                card.find('span', {'class': 'date'})
            )
            job['date'] = date_elem.text.strip() if date_elem else 'N/A'
            
            # Description extraite (snippet)
            snippet_elem = card.find('div', {'data-testid': 'belowJobSnippet'})
            if snippet_elem:
                # Nettoyer le HTML
                for li in snippet_elem.find_all('li'):
                    if li.text.strip():
                        job.setdefault('description', []).append(li.text.strip())
            else:
                job['description'] = []
            
            # Badges (nouveau, candidature simplifiée, etc.)
            badges = []
            badge_containers = card.find_all('div', {'class': 'mosaic-provider-jobcards-1k6cgqp'})
            for container in badge_containers:
                for badge in container.find_all('div', {'class': 'mosaic-provider-jobcards-1f1q1js'}):
                    if badge.text.strip():
                        badges.append(badge.text.strip())
            job['badges'] = ', '.join(badges) if badges else 'N/A'
            
            # Entreprise sponsorisée ?
            sponsored = card.find('div', {'class': 'sponTapItem'}) is not None
            job['sponsorise'] = 'Oui' if sponsored else 'Non'
            
            # Job Key (identifiant unique)
            job_key_elem = card.find('a', {'data-jk': True})
            job['job_key'] = job_key_elem['data-jk'] if job_key_elem else 'N/A'

            base_job = 'https://ch-fr.indeed.com/viewjob?jk=' if job_key_elem else 'N/A'
            # Lien de l'offre
            job['lien'] = f'{base_job}{job_key_elem['data-jk']}'
            
            jobs.append(job)
            
        except Exception as e:
            print(f"⚠️ Erreur parsing d'une carte: {e}")
            continue
    
    return jobs

# Exécution du parsing
jobs_data = parse_indeed_jobs_advanced(html)

In [17]:
# Création du DataFrame
df = pd.DataFrame(jobs_data)

# Sauvegarde
if not df.empty:
    # Nettoyer les données pour l'export
    df_clean = df.copy()
    df_clean['description'] = df_clean['description'].apply(lambda x: ' | '.join(x) if isinstance(x, list) else x)
    df_clean = df_clean[[ len(str(x)) > 0 for x in df_clean["description"]]]

    print(f"{len(df_clean)} offres trouvées")

    # Sauvegarde CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"indeed_offres_{timestamp}.csv"
    df_clean.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"💾 Données sauvegardées dans {filename}")

51 offres trouvées
💾 Données sauvegardées dans indeed_offres_20260818_140801.csv
